<a href="https://colab.research.google.com/github/Bor7c/DL/blob/main/%D0%94%D0%971.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!mkdir -p /content/data
!unzip -q /content/Картинки.zip -d /content/data/

In [9]:
!mv /content/data/Больница /content/data/hospital
!mv /content/data/Аптека /content/data/pharmacy
!mv /content/data/Лаборатория /content/data/laboratory

In [29]:
from pathlib import Path
import numpy as np
from PIL import Image

# Используем английские имена, которые есть в папках
classes = ['hospital', 'pharmacy', 'laboratory']
IMG_SIZE = 224

def load_data_from_folders(base_path, classes, img_size):
    images = []
    labels = []
    for idx, cls in enumerate(classes):
        folder = Path(base_path) / cls
        if not folder.exists():
            raise FileNotFoundError(f"Папка {folder} не найдена")
        for img_path in folder.glob('*'):
            if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                continue
            img = Image.open(img_path).convert('RGB')
            img = img.resize((img_size, img_size), Image.LANCZOS)
            img_array = np.array(img, dtype=np.uint8)
            images.append(img_array)
            labels.append(idx)
    return np.array(images), np.array(labels)

# Путь к папке с данными
base_path = '/content/data'
X, y = load_data_from_folders(base_path, classes, IMG_SIZE)
print(f'Всего изображений: {len(X)}')
print(f'Распределение по классам: {np.bincount(y)}')

Всего изображений: 300
Распределение по классам: [100 100 100]


In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torchvision.models as models
from torchvision.models import MobileNet_V2_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import clear_output
import matplotlib.pyplot as plt
import os

In [34]:
EPOCHS = 40
BATCH_SIZE = 32
LR = 0.001
WEIGHT_DECAY = 1e-5
OPTIMIZER_NAME = "Adam"
LABEL_SMOOTHING = 0.2
AUG_PROB = 0.5
IMG_SIZE = 224
CLASSES = ['hospital', 'pharmacy', 'laboratory']
DATA_PATH = Path("/content/data")

In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется устройство: {device}")

Используется устройство: cpu


In [36]:
# ==================== 4. Загрузка данных ====================
def load_data_from_folders(base_path, classes, img_size):
    images, labels = [], []
    for idx, cls in enumerate(classes):
        folder = base_path / cls
        if not folder.exists():
            print(f"Предупреждение: папка {folder} не найдена!")
            continue
        for img_path in folder.iterdir():
            if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                continue
            try:
                img = Image.open(img_path).convert('RGB')
                img = img.resize((img_size, img_size), Image.LANCZOS)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(idx)
            except Exception as e:
                print(f"Ошибка загрузки {img_path}: {e}")
    return np.array(images), np.array(labels)

X, y = load_data_from_folders(DATA_PATH, CLASSES, IMG_SIZE)
print(f"Всего изображений: {len(X)}")
print(f"Распределение по классам: {np.bincount(y)}")

Всего изображений: 300
Распределение по классам: [100 100 100]


In [37]:
# ==================== 5. Разделение на train/test ====================
train_X, test_X, train_y, test_y = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(train_X)} изобр., Test: {len(test_X)} изобр.")

Train: 240 изобр., Test: 60 изобр.


In [38]:
# ==================== 6. Аугментация и датасеты ====================
class CustomDataset(Dataset):
    def __init__(self, X, y, transform=None, aug_prob=0.5):
        super().__init__()
        self.X = torch.FloatTensor(X)          # (N, H, W, C)
        self.y = torch.LongTensor(y)
        self.transform = transform
        self.aug_prob = aug_prob

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].permute(2, 0, 1)       # (C, H, W)
        if self.transform and np.random.random() < self.aug_prob:
            x = self.transform(x / 255.0) * 255.0
        return x, self.y[idx]

train_transform = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
])

train_dataset = CustomDataset(train_X, train_y, transform=train_transform, aug_prob=AUG_PROB)
test_dataset = CustomDataset(test_X, test_y, transform=None, aug_prob=0.0)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [39]:
# ==================== 7. Модель и нормализация ====================
class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        x = x / 255.0
        return (x - self.mean) / self.std

mobilenet = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
num_features = mobilenet.classifier[1].in_features
mobilenet.classifier[1] = nn.Linear(num_features, len(CLASSES))

model = nn.Sequential(
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    mobilenet
).to(device)

In [40]:
# Заморозка всех слоёв, кроме классификатора
for param in model[1].features.parameters():
    param.requires_grad = False
for param in model[1].classifier.parameters():
    param.requires_grad = True


In [41]:
# ==================== 8. Оптимизатор, функция потерь, планировщик ====================
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.RMSprop(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

In [42]:
# ==================== 9. Обучение ====================
epochs = EPOCHS
redraw_every = 5
train_losses, train_accs = [], []
test_losses, test_accs = [], []
best_acc = 0.0
best_epoch = 0

pbar = tqdm(total=epochs * len(train_loader), desc="Обучение")
for epoch in range(epochs):
    # --- Train ---
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
        pbar.update(1)

    epoch_loss = running_loss / total
    epoch_acc = correct / total * 100
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, pred = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (pred == labels).sum().item()

    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total * 100
    test_losses.append(val_epoch_loss)
    test_accs.append(val_epoch_acc)
    scheduler.step()

    if val_epoch_acc > best_acc:
        best_acc = val_epoch_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), 'best_model.pth')

    if (epoch + 1) % redraw_every == 0:
        clear_output(wait=True)
        print(f'Эпоха {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2f}% | Val Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.2f}%')
        print(f'Лучшая точность: {best_acc:.2f}% (эпоха {best_epoch})')
        # Графики (код опущен для краткости, он такой же, как в исходном)
        pbar = tqdm(total=epochs * len(train_loader))
        pbar.update((epoch + 1) * len(train_loader))

# Загрузка лучшей модели
model.load_state_dict(torch.load('best_model.pth'))
print(f'\n✅ Обучение завершено! Лучшая точность: {best_acc:.2f}% на эпохе {best_epoch}')


Обучение:   0%|          | 0/320 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ==================== 10. Оценка на train и test ====================
def evaluate(loader, description):
    y_true, y_pred = [], []
    model.eval()
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs).cpu().numpy()
            y_pred.extend(np.argmax(outputs, axis=1))
            y_true.extend(labels.numpy())
    print(f"\n{description}")
    print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
    return classification_report(y_true, y_pred, target_names=CLASSES, digits=4, output_dict=True)

train_report = evaluate(train_loader, "Результаты на ОБУЧАЮЩЕЙ выборке:")
test_report = evaluate(test_loader, "Результаты на ТЕСТОВОЙ выборке:")


In [ ]:
# ==================== 11. Вывод таблицы с конфигурацией и итогами ====================
print("\n" + "="*70)
print("ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*70)
print(f"Модель: MobileNetV2 (предобученная на ImageNet)")
print(f"Классы: {', '.join(CLASSES)}")
print(f"Гиперпараметры: lr = {LR}, batch_size = {BATCH_SIZE}, epochs = {EPOCHS}, weight_decay = {WEIGHT_DECAY}")
print(f"Оптимизатор: {OPTIMIZER_NAME}, label_smoothing = {LABEL_SMOOTHING}, аугментация с вероятностью {AUG_PROB}")
print(f"Точность на обучающей выборке: {train_report['accuracy']*100:.2f}%")
print(f"Точность на тестовой выборке: {test_report['accuracy']*100:.2f}%")
print("\nДетали по классам (тестовая выборка):")
for cls in CLASSES:
    print(f"{cls:15} Precision: {test_report[cls]['precision']:.4f}  Recall: {test_report[cls]['recall']:.4f}  F1-score: {test_report[cls]['f1-score']:.4f}")

In [ ]:
# ==================== 12. Экспорт в ONNX ====================
model_cpu = model.cpu().eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_filename = "model.onnx"
torch.onnx.export(model_cpu, dummy_input, onnx_filename,
                  export_params=True, opset_version=12,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})
print(f"\n✅ Модель экспортирована в {onnx_filename}")
